# Candidate Generation: Static Item-Item Collaborative Filtering

Builds a static binary user-item matrix from the training split, derives
item-item top-K cosine co-occurrence neighbours, and writes validation Top-500
candidates in the standard candidate format.

Protocol:

- The matrix uses only training data: latest `full_history_rows` per user plus all
  `candidates_train` clicks, deduplicated to binary interactions.
- `article_row` is the matrix column key; `news_code` is written only to candidate files.
- Validation requests are scored causally; history articles are excluded.
- Eligibility is `available_from <= timestamp`.
- Empty-history requests fall back to popularity.

`REBUILD=True` rebuilds the matrix, neighbours and candidates. The notebook
runs the full pipeline by default; set `FULL_RUN=False` in the setup cell to
score a request subset, write `*_smoke` artifacts and compute subset recall
instead of the shared evaluator. `CF_DEVICE="auto"` uses CUDA for the
item-item top-K when available.

## 1. Setup

In [ ]:
from pathlib import Path
import heapq
import json
import sys
from datetime import datetime, timezone

import numpy as np
import polars as pl
from scipy import sparse

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))

from cf_models import build_interaction_matrix, load_context, resolve_device
from cg_result_writer import CandidateWriter

CANDIDATE_CAP = 500
ITEM_K = 100
DEFAULT_M_VALUES = (50, 100, 200, 500)
REBUILD = False
FULL_RUN = True                # Toggle here: False runs a smoke pipeline.
RUN_TAG = "cf_item_item" if FULL_RUN else "cf_item_item_smoke"
SMOKE_REQUESTS = 3_000
CF_DEVICE = "auto"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

if not FULL_RUN and SMOKE_REQUESTS < 1:
    raise ValueError("SMOKE_REQUESTS must be positive")

print("Repository:", ROOT)
print("Candidate cap:", CANDIDATE_CAP)
print("Item K:", ITEM_K)
print("Device:", CF_DEVICE)
print("Mode:", "full" if FULL_RUN else "smoke")

## 2. Data ingestion and model-related processing

### Static user-item matrix

In [ ]:
context = load_context(ROOT)
matrix, matrix_manifest = build_interaction_matrix(
    ROOT, context, rebuild=REBUILD
)

print(f"Matrix shape: {matrix.shape[0]:,} users x {matrix.shape[1]:,} articles")
print(f"Nonzeros: {matrix.nnz:,}")
print(f"Train cutoff: {matrix_manifest['cutoff']}")
matrix_manifest

## 3. Setting up the model

### Item-item similarity and top-K definitions

In [ ]:
def _normalised_columns(matrix):
    counts = np.asarray(matrix.sum(axis=0)).ravel().astype(np.float32)
    inverse = np.zeros_like(counts)
    nonzero = counts > 0
    inverse[nonzero] = 1.0 / np.sqrt(counts[nonzero])
    return matrix.T.tocsr().multiply(inverse[:, None]).tocsr(), counts

In [ ]:
def _to_torch_csr(matrix, device):
    import warnings

    import torch

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message="Sparse CSR tensor support is in beta state"
        )
        warnings.filterwarnings(
            "ignore", message="Sparse invariant checks are implicitly disabled"
        )
        return torch.sparse_csr_tensor(
            torch.from_numpy(matrix.indptr.astype(np.int64)).to(device),
            torch.from_numpy(matrix.indices.astype(np.int64)).to(device),
            torch.from_numpy(matrix.data.astype(np.float32)).to(device),
            size=matrix.shape,
            device=device,
            check_invariants=False,
        )

In [ ]:
def _top_k_block_gpu(matrix, k, block, device, self_exclude):
    import torch

    device = resolve_device(device)
    if device == "cpu":
        return _top_k_block(matrix, k, block, self_exclude)

    rows = matrix.shape[0]
    transpose = _to_torch_csr(matrix.T.tocsr(), device)
    row_parts, col_parts, value_parts = [], [], []
    with torch.no_grad():
        for start in range(0, rows, block):
            stop = min(start + block, rows)
            block_matrix = _to_torch_csr(matrix[start:stop], device)
            scores = torch.sparse.mm(block_matrix, transpose).to_dense()
            width = stop - start
            if self_exclude:
                local = torch.arange(width, device=device)
                scores[local, torch.arange(start, stop, device=device)] = -1.0
            values, indices = torch.topk(scores, k, dim=1)
            values = values.cpu().numpy()
            indices = indices.cpu().numpy()
            keep = values > 0
            if not keep.any():
                continue
            row_index = np.broadcast_to(
                np.arange(start, stop, dtype=np.int32)[:, None], keep.shape
            )[keep]
            row_parts.append(row_index)
            col_parts.append(indices[keep].astype(np.int32, copy=False))
            value_parts.append(values[keep].astype(np.float32, copy=False))
            del block_matrix, scores, values, indices

    del transpose
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )

In [ ]:
def _top_k_block(matrix, k, block, self_exclude):
    rows = matrix.shape[0]
    row_parts, col_parts, value_parts = [], [], []
    for start in range(0, rows, block):
        stop = min(start + block, rows)
        scores = (matrix[start:stop] @ matrix.T).tocsr()
        scores.sum_duplicates()
        scores.eliminate_zeros()
        for local in range(scores.shape[0]):
            lo, hi = scores.indptr[local], scores.indptr[local + 1]
            cols = scores.indices[lo:hi]
            values = scores.data[lo:hi]
            if self_exclude and cols.size:
                keep = cols != (start + local)
                cols = cols[keep]
                values = values[keep]
            if cols.size == 0:
                continue
            if cols.size > k:
                chosen = np.argpartition(values, -k)[-k:]
                cols = cols[chosen]
                values = values[chosen]
            order = np.argsort(-values, kind="stable")
            cols = cols[order]
            values = values[order]
            row_parts.append(np.full(cols.size, start + local, dtype=np.int32))
            col_parts.append(cols.astype(np.int32, copy=False))
            value_parts.append(values.astype(np.float32, copy=False))
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )

In [ ]:
def build_item_neighbors(matrix, k=ITEM_K, block=256, device="auto"):
    normalised, popularity = _normalised_columns(matrix)
    if resolve_device(device) == "cuda":
        cosine = _top_k_block_gpu(
            normalised, k, block, device=device, self_exclude=True
        )
    else:
        cosine = _top_k_block(normalised, k, block, self_exclude=True)
    rows = np.repeat(
        np.arange(cosine.shape[0], dtype=np.int32), np.diff(cosine.indptr)
    )
    counts = cosine.copy()
    counts.data = counts.data * np.sqrt(
        popularity[rows] * popularity[cosine.indices]
    )
    return cosine, counts, popularity

## 4. Training the model

### Build and cache item-item neighbours

In [ ]:
def load_or_build_item_neighbors(
    root, matrix, k=ITEM_K, *, rebuild=False, device="auto"
):
    output = (
        Path(root)
        / "5_Models"
        / "5_01_Candidate_Generation"
        / "artifacts"
        / "cf_item_item"
    )
    output.mkdir(parents=True, exist_ok=True)
    cosine_path = output / "item_item_cosine_topk.npz"
    counts_path = output / "item_item_counts_topk.npz"
    manifest_path = output / "item_item_manifest.json"

    if (
        cosine_path.exists()
        and counts_path.exists()
        and manifest_path.exists()
        and not rebuild
    ):
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("k") == k and manifest.get("n_items") == matrix.shape[1]:
            return (
                sparse.load_npz(cosine_path),
                sparse.load_npz(counts_path),
                np.asarray(np.load(output / "item_popularity.npy")),
                manifest,
            )

    resolved = resolve_device(device)
    cosine, counts, popularity = build_item_neighbors(
        matrix, k=k, device=resolved
    )
    sparse.save_npz(cosine_path, cosine, compressed=True)
    sparse.save_npz(counts_path, counts, compressed=True)
    np.save(output / "item_popularity.npy", popularity)
    manifest = {
        "version": "cf-item-item-v1",
        "created_at": _now(),
        "k": int(k),
        "n_items": int(matrix.shape[1]),
        "edges": int(cosine.nnz),
        "device": resolved,
        "similarity": "cosine over binary user vectors",
        "counts": "raw co-occurrence derived from cosine and item popularity",
    }
    _write_json(manifest_path, manifest)
    return cosine, counts, popularity, manifest

In [ ]:
item_similarity, item_counts, item_popularity, item_manifest = (
    load_or_build_item_neighbors(
        ROOT, matrix, k=ITEM_K, rebuild=REBUILD, device=CF_DEVICE
    )
)

print(f"Item-item edges: {item_similarity.nnz:,}")
print(f"Items with neighbours: {(np.diff(item_similarity.indptr) > 0).sum():,}")
item_manifest

## 5. Results

### Generate validation candidates

In [ ]:
def _now():
    return datetime.now(timezone.utc).isoformat()

In [ ]:
def _write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n")

In [ ]:
def _load_profiles(root, split):
    root = Path(root)
    features = root / "4_Feature_Engineering" / "artifacts" / "prequential-all"
    parts = sorted((features / f"request_profiles_{split}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No request profiles for {split}")
    frame = pl.read_parquet(
        parts,
        columns=["impression_uid", "user_code", "timestamp", "full_history_rows"],
    )
    if frame["impression_uid"].n_unique() != frame.height:
        raise ValueError("Duplicate impression_uid in request profiles")
    if frame["full_history_rows"].null_count():
        raise ValueError("Null full_history_rows in request profiles")
    return frame.sort("timestamp", maintain_order=True)

In [ ]:
class PopularityFallback:
    def __init__(self, available_from, popularity, news_code, limit=1500):
        self.available_from = available_from
        self.popularity = popularity
        self.news_code = news_code
        self.limit = int(limit)
        order = np.argsort(available_from[1:], kind="stable") + 1
        self.order = order.astype(np.int64)
        self.order_available = available_from[self.order]
        self.position = 0
        self.heap = []

    def advance(self, timestamp):
        while (
            self.position < self.order.size
            and self.order_available[self.position] <= timestamp
        ):
            row = int(self.order[self.position])
            key = (
                float(self.popularity[row]),
                -int(self.news_code[row]),
                row,
            )
            if len(self.heap) < self.limit:
                heapq.heappush(self.heap, key)
            elif key > self.heap[0]:
                heapq.heapreplace(self.heap, key)
            self.position += 1
        items = sorted(self.heap, key=lambda x: (-x[0], -x[1]))
        return np.asarray([item[2] for item in items], dtype=np.int32)

In [ ]:
def _row_aggregate(matrix):
    matrix = matrix.tocsr()
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    if matrix.nnz == 0:
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)
    cols = matrix.indices
    values = matrix.data
    unique, inverse = np.unique(cols, return_inverse=True)
    aggregated = np.zeros(unique.size, dtype=np.float32)
    np.add.at(aggregated, inverse, values)
    return unique.astype(np.int32), aggregated

In [ ]:
def select_top_k(
    candidate_rows,
    candidate_scores,
    history,
    popularity,
    news_code,
    available_from,
    timestamp,
    fallback,
    k=CANDIDATE_CAP,
):
    candidate_rows = np.asarray(candidate_rows, dtype=np.int64)
    candidate_scores = np.asarray(candidate_scores, dtype=np.float32)
    history = np.asarray(history, dtype=np.int64)
    selected = np.empty(0, dtype=np.int64)

    if candidate_rows.size:
        keep = (
            (candidate_rows >= 1)
            & (candidate_rows < news_code.size)
            & (available_from[candidate_rows] <= timestamp)
        )
        if history.size:
            keep &= ~np.isin(candidate_rows, history)
        candidate_rows = candidate_rows[keep]
        candidate_scores = candidate_scores[keep]
        if candidate_rows.size:
            order = np.lexsort(
                (
                    news_code[candidate_rows],
                    -popularity[candidate_rows],
                    -candidate_scores,
                )
            )
            selected = candidate_rows[order[:k]]

    if selected.size < k:
        remaining = np.asarray(fallback, dtype=np.int64)
        if remaining.size:
            keep = available_from[remaining] <= timestamp
            if history.size:
                keep &= ~np.isin(remaining, history)
            if selected.size:
                keep &= ~np.isin(remaining, selected)
            remaining = remaining[keep]
        needed = k - selected.size
        if remaining.size < needed:
            raise RuntimeError("Not enough eligible popular fallback articles")
        selected = np.concatenate([selected, remaining[:needed]])

    return selected.astype(np.int32, copy=False)

In [ ]:
def _select_for_request(
    context,
    item_similarity,
    history,
    fallback,
    timestamp,
):
    history = np.asarray(history, dtype=np.int64)
    history = history[(history >= 1) & (history < context.n_items)]
    history = np.unique(history)
    if history.size == 0:
        candidate_rows = np.empty(0, dtype=np.int32)
        candidate_scores = np.empty(0, dtype=np.float32)
    else:
        candidate_rows, candidate_scores = _row_aggregate(
            item_similarity[history]
        )
    return select_top_k(
        candidate_rows,
        candidate_scores,
        history,
        context.popularity,
        context.news_code,
        context.available_from,
        timestamp,
        fallback,
    )

In [ ]:
def generate_recommendations(
    root,
    context,
    item_similarity,
    *,
    artifact_dir,
    limit=None,
    rebuild=False,
):
    output = Path(artifact_dir)
    output.mkdir(parents=True, exist_ok=True)
    mode = "full" if limit is None else "smoke"
    path = output / f"recommendations_validation_top{CANDIDATE_CAP}.parquet"
    manifest_path = output / "manifest_validation.json"

    if path.exists() and manifest_path.exists() and not rebuild:
        manifest = json.loads(manifest_path.read_text())
        if (
            manifest.get("model") == "cf_item_item"
            and manifest.get("mode", "full") == mode
        ):
            return path, manifest

    profiles = _load_profiles(root, "validation")
    uids = profiles["impression_uid"].to_numpy()
    timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    histories = profiles["full_history_rows"].to_list()
    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))

    if limit is not None:
        if limit < 1:
            raise ValueError("limit must be positive")
        batch_rng = np.random.default_rng(4222)
        batch_order = batch_rng.permutation(len(batches))
        selected_batches = []
        total = 0
        for index in batch_order:
            start, end = batches[index]
            selected_batches.append((start, end))
            total += end - start
            if total >= int(limit):
                break
        batches = sorted(selected_batches)

    processed_requests = int(sum(end - start for start, end in batches))
    print(f"Scoring requests: {processed_requests:,} / {len(timestamps):,}")

    fallback = PopularityFallback(
        context.available_from,
        context.popularity,
        context.news_code,
        limit=CANDIDATE_CAP + 1200,
    )
    cutoffs = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(path)
    try:
        for start, end in batches:
            timestamp = timestamps[start]
            fallback_rows = fallback.advance(timestamp)
            selected = [
                _select_for_request(
                    context,
                    item_similarity,
                    histories[row],
                    fallback_rows,
                    timestamp,
                )
                for row in range(start, end)
            ]
            selected_matrix = np.vstack(selected)
            writer.add(
                np.repeat(uids[start:end], CANDIDATE_CAP),
                np.tile(cutoffs, end - start),
                context.news_code[selected_matrix.reshape(-1)],
            )
    finally:
        writer.close()

    expected = processed_requests * CANDIDATE_CAP
    if writer.rows != expected:
        raise ValueError(f"Expected {expected} rows, wrote {writer.rows}")

    manifest = {
        "model": "cf_item_item",
        "created_at": _now(),
        "split": "validation",
        "candidate_cap": CANDIDATE_CAP,
        "candidate_format": ["impression_uid", "cutoff", "news_code"],
        "target_requests": processed_requests,
        "recommendation_rows": int(writer.rows),
        "eligibility": "available_from <= timestamp",
        "history_policy": "exclude every article in the request full_history_rows",
        "fallback": "popularity order among eligible articles",
        "matrix": "static binary user-item matrix from train",
        "recommendations_file": path.name,
        "scoring": "sum of top-K item-item cosine neighbours of history items",
        "k": ITEM_K,
        "mode": mode,
        "smoke_requests": limit,
        "smoke_recall_file": None,
    }
    _write_json(manifest_path, manifest)
    return path, manifest

In [ ]:
item_path, item_recommendations = generate_recommendations(
    ROOT,
    context,
    item_similarity,
    artifact_dir=ARTIFACT_DIR,
    limit=None if FULL_RUN else SMOKE_REQUESTS,
    rebuild=REBUILD,
)

print(f"Item-item rows: {item_recommendations['recommendation_rows']:,}")
print("Item-item file:", item_path)
item_recommendations

### Evaluate Recall@M

In [ ]:
def smoke_recall(recommendations_path, model_name, impression_uids):
    uid_frame = pl.DataFrame(
        {"impression_uid": np.asarray(impression_uids, dtype=np.int64)}
    )
    label_parts = sorted(
        (FEATURES / "labels_validation").glob("*.parquet")
    )
    if not label_parts:
        raise FileNotFoundError("No validation labels")
    labels = pl.read_parquet(
        label_parts,
        columns=["impression_uid", "news_code", "clicked"],
    ).join(uid_frame, on="impression_uid", how="inner")
    positives = labels.filter(pl.col("clicked") == 1).select(
        "impression_uid", "news_code"
    )
    relevant = positives.group_by("impression_uid").agg(
        pl.len().alias("relevant")
    )
    base = uid_frame.join(
        relevant, on="impression_uid", how="left"
    ).with_columns(pl.col("relevant").fill_null(0).cast(pl.Int64))
    candidates = pl.read_parquet(
        recommendations_path,
        columns=["impression_uid", "news_code", "cutoff"],
    )
    rows = []
    for m in DEFAULT_M_VALUES:
        selected = candidates.filter(pl.col("cutoff") <= m)
        hits = (
            positives.join(
                selected.select("impression_uid", "news_code"),
                on=["impression_uid", "news_code"],
                how="inner",
            )
            .group_by("impression_uid")
            .agg(pl.len().alias("hits"))
        )
        per_impression = base.join(
            hits, on="impression_uid", how="left"
        ).with_columns(pl.col("hits").fill_null(0).cast(pl.Int64))
        relevant_clicks = int(per_impression["relevant"].sum())
        retrieved_relevant = int(per_impression["hits"].sum())
        with_relevant = per_impression.filter(pl.col("relevant") > 0)
        recall_impression_mean = (
            float(
                (
                    with_relevant["hits"] / with_relevant["relevant"]
                ).mean()
            )
            if with_relevant.height
            else float("nan")
        )
        hit_rate = (
            float((per_impression["hits"] > 0).sum() / per_impression.height)
            if per_impression.height
            else float("nan")
        )
        rows.append(
            {
                "model": model_name,
                "split": "validation",
                "m": int(m),
                "recall_pooled": (
                    retrieved_relevant / relevant_clicks
                    if relevant_clicks
                    else float("nan")
                ),
                "recall_impression_mean": recall_impression_mean,
                "hit_rate": hit_rate,
                "impressions": int(per_impression.height),
                "relevant_clicks": relevant_clicks,
                "retrieved_relevant": retrieved_relevant,
                "mean_candidates": float(min(m, CANDIDATE_CAP)),
            }
        )
    return pl.DataFrame(rows)

In [ ]:
def evaluate_recommendations(root, paths, split="validation"):
    root = Path(root)
    from cg_evaluation import CandidateEvaluator

    evaluator = CandidateEvaluator(root=root, split=split)
    results = []
    for model, path in paths.items():
        evaluation = evaluator.evaluate(
            path, model_name=model, m_values=DEFAULT_M_VALUES
        )
        output = root / "results" / f"candidate_generation_{model}_{split}.csv"
        output.parent.mkdir(parents=True, exist_ok=True)
        evaluation.write_csv(output)
        results.append(evaluation)
    combined = pl.concat(results)
    comparison = root / "results" / f"candidate_generation_cf_comparison_{split}.csv"
    combined.write_csv(comparison)

    existing_path = root / "results" / f"candidate_generation_comparison_{split}.csv"
    if existing_path.exists():
        existing = pl.read_csv(existing_path)
        model_names = set(combined["model"].to_list())
        existing = existing.filter(~pl.col("model").is_in(model_names))
        pl.concat([existing, combined]).write_csv(existing_path)
    return combined

In [ ]:
if FULL_RUN:
    results = evaluate_recommendations(ROOT, {"cf_item_item": item_path})
    display(results.sort(["m", "model"]))
else:
    smoke_frame = pl.read_parquet(item_path, columns=["impression_uid"])
    smoke_uids = smoke_frame["impression_uid"].unique().to_numpy()
    smoke_evaluation = smoke_recall(item_path, "cf_item_item", smoke_uids)
    smoke_recall_path = ARTIFACT_DIR / "smoke_recall_validation.csv"
    smoke_evaluation.write_csv(smoke_recall_path)
    print(smoke_evaluation.sort("m"))
    print("Smoke recall:", smoke_recall_path)
    manifest_path = ARTIFACT_DIR / "manifest_validation.json"
    manifest = json.loads(manifest_path.read_text())
    manifest["smoke_recall_file"] = smoke_recall_path.name
    _write_json(manifest_path, manifest)
    print("Smoke run: shared evaluator skipped; subset recall written.")